In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir('/zhome/71/c/146676/main/')
import SimpleITK as sitk
from loaders import loader_XA_to_NA
import importlib
from helpers import module_auxiliary as ma
importlib.reload(loader_XA_to_NA)
import tifffile
from matplotlib_scalebar.scalebar import ScaleBar
from scipy.ndimage import median_filter # For extract edge function
from cil.framework import ImageGeometry, ImageData # For extract edge function
from cil.optimisation.operators import GradientOperator # For extract edge function
import matplotlib
matplotlib.use('Agg')

In [ ]:
NA_surf_volume_ = np.load('/dtu-compute/msaca/cache/NA_surf1.npy')
XA_surf_volume_ = np.load('/dtu-compute/msaca/cache/XA_surf1.npy')
nz, ny, nx = np.shape(XA_surf_volume_)
subvol = [0,nz,30, 40, 0, nx]

NA_surf_volume = NA_surf_volume_[subvol[0]:subvol[1],subvol[2]:subvol[3],subvol[4]:subvol[5]]
XA_surf_volume = XA_surf_volume_[subvol[0]:subvol[1],subvol[2]:subvol[3],subvol[4]:subvol[5]]

In [ ]:
path = '/dtu-compute/msaca/sliceA_eds/EDS_BB_A_raw_files/edsMg.tiff'
eds = tifffile.imread(path)
eds[eds>150] = 0
eds = np.fliplr(eds)
eds = eds[4000::2,::2]/100
XA_surf = XA_surf_volume_[:,37]

In [ ]:
_ , nx_XA = np.shape(XA_surf)
_ , nx_eds = np.shape(eds)

fixed = sitk.GetImageFromArray(XA_surf)
moving = sitk.GetImageFromArray(eds)
moving_NA = sitk.GetImageFromArray(XA_surf)
fixed.SetSpacing((1/nx_XA,1/nx_XA))
moving.SetSpacing((1/nx_eds,1/nx_eds))

size_fixed = fixed.GetSize()
size_moving = moving.GetSize()

spacing_fixed = fixed.GetSpacing()
spacing_moving = moving.GetSpacing()

# Compute the new origin (shift it to -N/2)
new_origin_fixed = [-0.5 * (size_fixed[i] - 1) * spacing_fixed[i] for i in range(len(size_fixed))]
new_origin_moving = [-0.5 * (size_moving[i] - 1) * spacing_moving[i] for i in range(len(size_moving))]


fixed.SetOrigin(new_origin_fixed)
moving.SetOrigin(new_origin_moving)

In [ ]:
factor = 2

transform = sitk.Transform(2, sitk.sitkIdentity)
# Get the original size and spacing of the image
size = moving.GetSize()
spacing = moving.GetSpacing()

# Calculate the new size (downsampling by factor)
new_size = [int(size[0] / factor), int(size[1] / factor)]
# Calculate the new spacing (enlarging the spacing to match the downsampled size)
new_spacing = [s * factor for s in spacing]

# Perform the resampling (using average interpolation for downsampling)
moving_d = sitk.Resample(moving,
                                new_size,
                                transform,
                                sitk.sitkLinear,  # BSpline interpolation is good for downsampling
                                moving.GetOrigin(),
                                new_spacing,
                                moving.GetDirection(),
                                0)  # 0 is the background value for the resampling


In [ ]:
def registrator(fixed, moving, thresholds = [0.025, 30], sampling_percentage = 0.1, max_iter = 1000, learning_rate = 1,mask=None):

    fixed_d = sitk.BinaryThreshold(fixed, lowerThreshold=thresholds[0], upperThreshold=float("inf"), insideValue=1, outsideValue=0)
    moving_d = sitk.BinaryThreshold(moving, lowerThreshold=thresholds[1], upperThreshold=float("inf"), insideValue=1, outsideValue=0)
    fixed_d =sitk.Cast(fixed_d, sitk.sitkFloat32)
    moving_d =sitk.Cast(moving_d, sitk.sitkFloat32)

    initial_transform = sitk.Similarity2DTransform()
    initial_transform.SetMatrix([1.0, 0.0,
                                0.0, 1.0])

    # Set the translation to zero
    initial_transform.SetTranslation([0.0, 0.0])
    initial_transform.SetScale(1.0)

    registration = sitk.ImageRegistrationMethod()
    registration.SetInitialTransform(initial_transform)
    registration.SetMetricAsMeanSquares()
    if mask is not None:
        mask = sitk.GetImageFromArray(mask)
        mask.CopyInformation(fixed_d) 
        registration.SetMetricFixedMask(mask)  # Apply mask to the fixed image

    registration.SetMetricSamplingStrategy(registration.RANDOM)
    registration.SetMetricSamplingPercentage(sampling_percentage)

    registration.SetOptimizerAsGradientDescent(
        learningRate=learning_rate,
        numberOfIterations=max_iter,
        convergenceMinimumValue=-1e-16,
        convergenceWindowSize=1000
        )

    registration.SetInterpolator(sitk.sitkLinear)
    registration.Execute(fixed_d, moving_d)
    transform = registration.GetInitialTransform()
    return transform

def resampler(fixed, moving, transform):
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(fixed)  # Reference image (fixed)
    resampler.SetInterpolator(sitk.sitkLinear)   # Interpolation method
    resampler.SetTransform(transform)    # Apply the initial transform (aligned centroids)
    resampler.SetOutputPixelType(fixed.GetPixelID())
    resampler.SetOutputSpacing(fixed.GetSpacing())  # Ensure the spacing is preserved
    resampler.SetOutputOrigin(fixed.GetOrigin())  # Preserve origin
    resampler.SetOutputDirection(fixed.GetDirection())
    moving = resampler.Execute(moving)
    return moving

In [ ]:
transforms = []

transforms.append(registrator(fixed, moving_d, thresholds = [0.020, 0.20], sampling_percentage = 0.4, max_iter = 200, learning_rate = 3))
moving_temp = resampler(fixed, moving_d, transforms[0])



transforms.append(registrator(fixed, moving_temp, thresholds = [0.020, 0.20], sampling_percentage = 0.4, max_iter = 200, learning_rate = 1))
moving_temp = resampler(fixed, moving_temp, transforms[1])

z_min, z_max = 300, 1000
x_min, x_max = 500, 1250
# Create an empty binary mask
mask_array = np.zeros((nz, nx), dtype=np.uint8)

# Set the region inside the bounding box to 1
mask_array[z_min:z_max, x_min:x_max,] = 1

transforms.append(registrator(fixed, moving_temp, thresholds = [0.024, 0.20], sampling_percentage = 0.4,
 max_iter = 200, learning_rate = 0.5,mask = mask_array))
moving_temp = resampler(fixed, moving_temp, transforms[2])



In [ ]:
plt.figure()
plt.imshow(sitk.GetArrayFromImage(moving_temp)[1100:1200,500:600])
plt.savefig('output.png', dpi=200)
plt.figure()
plt.imshow(sitk.GetArrayFromImage(fixed)[1100:1200,500:600])
plt.savefig('output0.png', dpi=200)


In [ ]:
# Load images

# Define 3 corresponding landmark points (in **index coordinates**)
fixed_points = np.array([
    [325, 468],  # (row, column) format
    [1273, 1150],
    [363, 1187],
    [1160, 595]
], dtype=np.float64)

moving_points = np.array([
    [332, 461],  # Matching indices in moving image
    [1260, 1159],
    [367, 1190],
    [1154,593]
], dtype=np.float64)

# Convert index coordinates to physical space (important for SimpleITK)
def indices_to_physical(image, points):
    return [coord for p in points for coord in image.TransformIndexToPhysicalPoint([int(p[1]), int(p[0])])]


fixed_landmarks = indices_to_physical(fixed, fixed_points)
moving_landmarks = indices_to_physical(moving_temp, moving_points)

# Initialize a transformation using the landmark points
transform = sitk.AffineTransform(2)  # You can also use AffineTransform(2)
transform = sitk.LandmarkBasedTransformInitializer(transform, fixed_landmarks, moving_landmarks)
transforms.append(transform)
moving_temp = resampler(fixed, moving_temp, transform)


In [ ]:
m1 = sitk.GetArrayFromImage(moving_temp)>0.2
m2 = sitk.GetArrayFromImage(fixed)>0.024
plt.figure()
plt.imshow(m1*1-m2*1)
plt.savefig('output.png', dpi=200)
plt.figure()
plt.imshow(m1*1)
plt.savefig('output0.png', dpi=200)
plt.figure()
plt.imshow(m2*1)
plt.savefig('output00.png', dpi=200)

In [ ]:

transformation_history = sitk.CompositeTransform(2)
for i in range(len(transforms)):
    transformation_history.AddTransform(transforms[i])
#path = 'transformation_EDS_to_NA.tfm'
#sitk.WriteTransform(transformation_history, path)


In [ ]:
from segmentation import s2_watershed_filter
importlib.reload(s2_watershed_filter)
subvol = [0,nz,30, 40, 0, nx]
XA_sub = XA_surf_volume_[subvol[0]:subvol[1],subvol[2]:subvol[3],subvol[4]:subvol[5]]
NA_sub = NA_surf_volume_[subvol[0]:subvol[1],subvol[2]:subvol[3],subvol[4]:subvol[5]]
pwc_XA, pwc_NA = s2_watershed_filter.place_medians_in_watersheds(XA_sub, NA_sub,
    eta_edge = 0.005,level = 0.15, conductivity = 0.5, smoothing_iter = 15, watershed_line = False)

In [ ]:
## Materials:
# High xray low neutron: Zirconium


In [ ]:
plt.imshow(pwc_XA[:,5])
plt.clim([-0.01,0.07])
plt.savefig('output.png', dpi=250)

In [ ]:
from segmentation import s3_watershed_eds
importlib.reload(s3_watershed_eds)
full_seg_, segmentation_, elem_segm_ = s3_watershed_eds.get_segmentation_from_watersheds(thresholds = np.array([15, 15, 15, 10 , 20, 10, 30, 12, 5 , 30, 10, 13 , 92, 10, 5]),
    output=True)
plt.imshow(full_seg_)
plt.savefig('output.png', dpi=200) 


In [ ]:
segmentation = {}
for key in segmentation_:
    if key != "labels":
        
        moving = sitk.GetImageFromArray(segmentation_[key].astype(np.float32))
        segmentation[key] = s3_watershed_eds.transform_eds(segmentation_[key].astype(np.float32))

moving = sitk.GetImageFromArray(full_seg_)
full_seg = s3_watershed_eds.transform_eds(full_seg_)

In [ ]:
plt.imshow(segmentation['baddelyite']*1.0)
plt.savefig('plots/output.png',dpi=200)

In [ ]:
plt.imshow(full_seg)
plt.savefig('output.png', dpi=200)

In [ ]:
watershedded = np.load('/dtu-compute/msaca/cache/eds_watershedded.npy')
plt.imshow(watershedded[0,:,:])
plt.savefig('output.png', dpi=200)

In [ ]:
import matplotlib.colors as mcolors
# Add black as the first color
colors_d = [
    (0, 0, 255),      # Blue Plagio
    (128, 0, 128),    # Purple Alk
    (255, 50, 150),   # Pink   #cpx
    (0, 255, 0),      # Green   # opx
    (255, 165, 0),    # Orange     #apatite
    (255, 255, 0),    # Yellow Ilminite
    (0, 170, 255),     # light blue, chromite
    (255, 0, 0),    #  pyrite   
    (0, 125, 0),   # Magenta, Baddelyite
]
#colors = [(0, 0, 0)] + colors
# Create a ListedColormap
colors_d = [(0, 0, 0)] + [(r/255, g/255, b/255) for r, g, b in colors_d]
cmap_d = mcolors.ListedColormap(colors_d)
boundaries = np.arange(-0.5, 10, 1)  # [-0.5, 0.5, 1.5, ..., 14.5]
norm = mcolors.BoundaryNorm(boundaries, cmap_d.N)
# Plot the matrix with the custom colormap

In [ ]:
plt.close('all')
fig, ax = plt.subplots(figsize=(10, 10))

full_seg[]

im = ax.imshow(full_seg, cmap=cmap_d, norm=norm,  alpha=0.7)

# Create a colorbar next to the plot
cbar = fig.colorbar(im, ax=ax, ticks=[], fraction=0.05, pad=0.04)
bounds = np.arange(len(colors_d) + 1)  # Define color boundaries
# Manually add text labels next to the colorbar
class_labels=['Background', 'Feldspar an-al', 'Feldspar al-or', 'Pyroxene (Cpx)', 'Pyroxene (Opx)', 'Apatite', 'Ilminite', 'Chromite', 'Pyrite', 'Baddelyite']
cbar_ax = cbar.ax  # Get the colorbar axis
for i, label in enumerate(class_labels):
    y_pos = (bounds[i] + bounds[i + 1]-1) / 2  # Center text at each color
    cbar_ax.text(1.3, y_pos, label, va='center', ha='left', fontsize=12)

cbar_ax.set_frame_on(False)  # Remove border

# Overlay the grayscale image using transparency
ax.imshow(pwc_XA[:,7,:], cmap="gray", alpha=0.5, vmin = -0.01, vmax = 0.07)  # Adjust alpha for blending

#ax.axis('off')
# Add scale bar (adjust values based on real-world scale)
scalebar = ScaleBar(1.82, "µm", location="lower right", color="white", scale_loc="bottom", box_alpha=0.5)
ax.add_artist(scalebar)
plt.tight_layout()
plt.savefig('plots/output_overlay.png', bbox_inches='tight', dpi = 250)

In [ ]:
plt.close('all')
m1 = segmentation['feldspar_an_al']
m2 = pwc_XA[:,5,:]>0.036
m3 = pwc_NA[:,5,:]<0.02
plt.figure(figsize=(10, 10))
plt.imshow(m1*1)
plt.savefig('output.png', dpi=200)
plt.figure(figsize=(10, 10))
plt.imshow(m2*1)
plt.savefig('output0.png', dpi=200)
plt.figure(figsize=(10, 10))
plt.imshow(m3*1)
plt.savefig('output00.png', dpi=200)
plt.figure(figsize=(10, 10))
plt.imshow(m3*m2*1)
plt.savefig('output000.png', dpi=200)

In [ ]:
import matplotlib.colors as mcolors
# Add black as the first color
colors_d = [
    (0, 0, 255),      # Blue Plagio
    (128, 0, 128),    # Purple Alk
    (255, 50, 150),   # Pink   #cpx
    (0, 255, 0),      # Green   # opx
    (255, 165, 0),    # Orange     #apatite
    (255, 255, 0),    # Yellow Ilminite
    (0, 170, 255),     # light blue, chromite
    (255, 0, 0),    #  pyrite   
    (255, 0, 255),   # Magenta, Baddelyite
]
#colors = [(0, 0, 0)] + colors
# Create a ListedColormap
colors_d = [(0, 0, 0)] + [(r/255, g/255, b/255) for r, g, b in colors_d]
cmap_d = mcolors.ListedColormap(colors_d)
boundaries = np.arange(-0.5, 7, 1)  # [-0.5, 0.5, 1.5, ..., 14.5]
norm = mcolors.BoundaryNorm(boundaries, cmap_d.N)
# Plot the matrix with the custom colormap
plt.close('all')
fig, ax = plt.subplots(figsize=(10, 10))

full_seg_special = np.maximum(full_seg-4,np.zeros(np.shape(full_seg)))
#full_seg_special[m2*m3] = 6
im = ax.imshow(full_seg_special, cmap=cmap_d, norm=norm,  alpha=0.7)

# Create a colorbar next to the plot
cbar = fig.colorbar(im, ax=ax, ticks=[], fraction=0.05, pad=0.04)
bounds = np.arange(len(colors_d) + 1)  # Define color boundaries
# Manually add text labels next to the colorbar
class_labels=['Background', 'Apatite', 'Ilminite', 'Chromite', 'Pyrite', 'Baddelyite', 'High_X_low_N']
cbar_ax = cbar.ax  # Get the colorbar axis
for i, label in enumerate(class_labels):
    y_pos = (bounds[i] + bounds[i + 1]-1) / 2  # Center text at each color
    cbar_ax.text(1.3, y_pos, label, va='center', ha='left', fontsize=12)

cbar_ax.set_frame_on(False)  # Remove border

# Overlay the grayscale image using transparency
ax.imshow(pwc_XA[:,7,:], cmap="gray", alpha=0.5, vmin = -0.01, vmax = 0.07)  # Adjust alpha for blending

#ax.axis('off')
# Add scale bar (adjust values based on real-world scale)
scalebar = ScaleBar(1.82, "µm", location="lower right", color="white", scale_loc="bottom", box_alpha=0.5)
ax.add_artist(scalebar)
plt.tight_layout()
plt.savefig('plots/output_overlay4.png', bbox_inches='tight', dpi = 250)